# 00f — Indian Classical Transcription (Colab T4)

Transcribes the remaining Hindustani and Carnatic tracks that were skipped in the
original local prep notebooks (which capped at 60 each due to CPU time).

**Run this on a Colab T4 GPU — Basic-Pitch inference is ~8-10x faster than CPU.**

| Tradition | Already done | Available | New to transcribe | Est. time (T4) |
|---|---|---|---|---|
| Hindustani | 60 | 108 | 48 | ~25 min |
| Carnatic | 60 | 149 (concert mixes) | 89 | ~45 min |

New MIDI files are saved directly to your Drive and appended to the existing
metadata CSVs. The training notebooks pick them up automatically on the next run.

**Note:** Carnatic tracks filtered to concert mixes only (files whose stem matches
their parent directory name). Separated stems (vocal-only, violin-only) are skipped.

## Cell 1 — Install Basic-Pitch

In [ ]:
import subprocess, sys, numpy as np

# Re-register numpy so pip can use it as a build backend
print(f'Re-registering numpy {np.__version__}...')
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                       f'numpy=={np.__version__}', '--force-reinstall', '--no-build-isolation'])
print('numpy OK')

# Use get_ipython().system() == the ! operator: Jupyter captures stdout+stderr and displays it
ip = get_ipython()
print('Installing basic-pitch (full output below)...')
ip.system('pip install basic-pitch --no-build-isolation')

import torch
print('CUDA:', torch.cuda.is_available(), '|',
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A')

from basic_pitch.inference import predict
from basic_pitch import ICASSP_2022_MODEL_PATH
print('Basic-Pitch ready:', ICASSP_2022_MODEL_PATH)

## Cell 2 — Mount Drive and Set Paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import pandas as pd
import re, time, warnings
warnings.filterwarnings('ignore')

THESIS_DATA = Path('/content/drive/MyDrive/Thesis-Data')

# Raw audio
HIND_RAW  = THESIS_DATA / 'datasets' / 'indian_classical' / 'saraga1.5_hindustani'
CARN_RAW  = THESIS_DATA / 'datasets' / 'indian_classical' / 'saraga1.5_carnatic'

# Output MIDI directories
HIND_MIDI = THESIS_DATA / 'data' / 'processed' / 'hindustani' / 'midi'
CARN_MIDI = THESIS_DATA / 'data' / 'processed' / 'carnatic'   / 'midi'
HIND_MIDI.mkdir(parents=True, exist_ok=True)
CARN_MIDI.mkdir(parents=True, exist_ok=True)

# Metadata CSVs
META_DIR  = THESIS_DATA / 'data' / 'metadata'
HIND_META = META_DIR / 'hindustani_tracks.csv'
CARN_META = META_DIR / 'carnatic_tracks.csv'

print('Hindustani raw  :', HIND_RAW.exists(), HIND_RAW)
print('Carnatic raw    :', CARN_RAW.exists(), CARN_RAW)
print('Hindustani MIDI :', HIND_MIDI, f'({len(list(HIND_MIDI.glob("*.mid")))} files)')
print('Carnatic MIDI   :', CARN_MIDI, f'({len(list(CARN_MIDI.glob("*.mid")))} files)')

## Cell 3 — Shared Transcription Helper

In [ ]:
from basic_pitch.inference import predict
from basic_pitch import ICASSP_2022_MODEL_PATH

def transcribe_audio(audio_path: Path, out_path: Path) -> bool:
    """Run Basic-Pitch on one audio file, save MIDI to out_path. Returns True on success."""
    try:
        _, midi_data, _ = predict(str(audio_path), ICASSP_2022_MODEL_PATH)
        midi_data.write(str(out_path))
        return True
    except Exception as e:
        print(f'    ERROR: {e}')
        return False


def safe_stem(path: Path) -> str:
    """File stem with double .mp3 extension removed."""
    name = path.name
    if name.endswith('.mp3.mp3'):
        name = name[:-8]
    elif name.endswith('.mp3'):
        name = name[:-4]
    return re.sub(r'[^\w\-]', '_', name)[:50]


def is_concert_mix(audio_path: Path) -> bool:
    """True if the audio file is a concert mix (stem matches parent dir name)."""
    stem = audio_path.name
    for ext in ['.mp3.mp3', '.mp3', '.wav']:
        if stem.endswith(ext):
            stem = stem[:-len(ext)]
    return stem == audio_path.parent.name


print('Helpers ready.')

## Cell 4 — Transcribe Remaining Hindustani Tracks

Finds all audio files in the Saraga Hindustani directory, skips any whose
audio path already appears in `hindustani_tracks.csv`, and transcribes the rest.

In [ ]:
import json

# ── Load existing metadata to know what's already done ────────────────────────
hind_meta = pd.read_csv(HIND_META) if HIND_META.exists() else pd.DataFrame()
already_done_hind = set(hind_meta['audio_path'].dropna().tolist()) if 'audio_path' in hind_meta.columns else set()
print(f'Already transcribed : {len(already_done_hind)} Hindustani tracks')

# ── Find all audio files ──────────────────────────────────────────────────────
all_hind_audio = sorted(
    list(HIND_RAW.rglob('*.mp3.mp3')) +
    [f for f in HIND_RAW.rglob('*.mp3') if not str(f).endswith('.mp3.mp3')]
)
print(f'Total audio files   : {len(all_hind_audio)}')

todo = [f for f in all_hind_audio if str(f) not in already_done_hind]
print(f'To transcribe       : {len(todo)}')

# ── Get next available index ──────────────────────────────────────────────────
existing_midis = sorted(HIND_MIDI.glob('hindustani_*.mid'))
start_idx = len(existing_midis)  # continue numbering from where we left off

# ── Transcribe ────────────────────────────────────────────────────────────────
new_rows = []
succeeded, failed_list = 0, []

for i, audio_path in enumerate(todo):
    global_idx = start_idx + i
    stem       = safe_stem(audio_path)
    out_name   = f'hindustani_{global_idx:03d}_{stem}.mid'
    out_path   = HIND_MIDI / out_name

    if out_path.exists():
        print(f'[{i+1:3d}/{len(todo)}] SKIP  {out_name}')
        new_rows.append({'audio_path': str(audio_path), 'midi_path': str(out_path)})
        succeeded += 1
        continue

    t0 = time.time()
    print(f'[{i+1:3d}/{len(todo)}] Transcribing: {audio_path.name} ...', end=' ', flush=True)
    ok = transcribe_audio(audio_path, out_path)
    elapsed = time.time() - t0

    if ok:
        # Try to extract raga from accompanying JSON
        json_path = audio_path.parent / (audio_path.parent.name + '.json')
        raga = None
        if json_path.exists():
            try:
                meta = json.loads(json_path.read_text())
                ragas = meta.get('raags') or meta.get('ragas') or []
                raga  = ragas[0].get('name') if ragas else None
            except Exception:
                pass

        new_rows.append({
            'audio_path': str(audio_path),
            'midi_path' : str(out_path),
            'raga'      : raga,
            'source'    : 'saraga_hindustani',
        })
        succeeded += 1
        print(f'done ({elapsed:.0f}s)')
    else:
        failed_list.append(str(audio_path))
        print(f'FAILED ({elapsed:.0f}s)')

print(f'\n=== Hindustani transcription done ===')
print(f'New tracks succeeded : {succeeded}')
print(f'Failed               : {len(failed_list)}')

# ── Append new rows to metadata CSV ──────────────────────────────────────────
if new_rows:
    new_df = pd.DataFrame(new_rows)
    updated = pd.concat([hind_meta, new_df], ignore_index=True)
    updated.to_csv(HIND_META, index=False)
    print(f'Metadata updated: {len(updated)} total rows → {HIND_META}')

total_hind = len(list(HIND_MIDI.glob('*.mid')))
print(f'Total Hindustani MIDI files now: {total_hind}')

## Cell 5 — Transcribe Remaining Carnatic Tracks

Saraga Carnatic contains both concert mixes and separated stems (vocal-only,
violin-only, etc.). We only want concert mixes — detected by checking whether
the audio file stem matches its parent directory name.

In [ ]:
# ── Load existing metadata ─────────────────────────────────────────────────────
carn_meta = pd.read_csv(CARN_META) if CARN_META.exists() else pd.DataFrame()
already_done_carn = set(carn_meta['audio_path'].dropna().tolist()) if 'audio_path' in carn_meta.columns else set()
print(f'Already transcribed : {len(already_done_carn)} Carnatic tracks')

# ── Find all concert-mix audio files ──────────────────────────────────────────
all_carn_audio = sorted(
    list(CARN_RAW.rglob('*.mp3.mp3')) +
    [f for f in CARN_RAW.rglob('*.mp3') if not str(f).endswith('.mp3.mp3')]
)
concert_mixes = [f for f in all_carn_audio if is_concert_mix(f)]
print(f'Total audio files   : {len(all_carn_audio)}')
print(f'Concert mixes only  : {len(concert_mixes)}')

todo = [f for f in concert_mixes if str(f) not in already_done_carn]
print(f'To transcribe       : {len(todo)}')

# ── Get next available index ──────────────────────────────────────────────────
existing_midis = sorted(CARN_MIDI.glob('carnatic_*.mid'))
start_idx = len(existing_midis)

# ── Transcribe ────────────────────────────────────────────────────────────────
new_rows = []
succeeded, failed_list = 0, []

for i, audio_path in enumerate(todo):
    global_idx = start_idx + i
    stem       = safe_stem(audio_path)
    out_name   = f'carnatic_{global_idx:03d}_{stem}.mid'
    out_path   = CARN_MIDI / out_name

    if out_path.exists():
        print(f'[{i+1:3d}/{len(todo)}] SKIP  {out_name}')
        new_rows.append({'audio_path': str(audio_path), 'midi_path': str(out_path)})
        succeeded += 1
        continue

    t0 = time.time()
    print(f'[{i+1:3d}/{len(todo)}] Transcribing: {audio_path.name} ...', end=' ', flush=True)
    ok = transcribe_audio(audio_path, out_path)
    elapsed = time.time() - t0

    if ok:
        # Try to extract raaga from accompanying JSON
        json_path = audio_path.parent / (audio_path.parent.name + '.json')
        raaga = None
        if json_path.exists():
            try:
                meta = json.loads(json_path.read_text())
                raagas = meta.get('raaga') or meta.get('ragas') or []
                raaga  = raagas[0].get('name') if raagas else None
            except Exception:
                pass

        new_rows.append({
            'audio_path': str(audio_path),
            'midi_path' : str(out_path),
            'raga'      : raaga,
            'source'    : 'saraga_carnatic',
        })
        succeeded += 1
        print(f'done ({elapsed:.0f}s)')
    else:
        failed_list.append(str(audio_path))
        print(f'FAILED ({elapsed:.0f}s)')

print(f'\n=== Carnatic transcription done ===')
print(f'New tracks succeeded : {succeeded}')
print(f'Failed               : {len(failed_list)}')

# ── Append new rows to metadata CSV ──────────────────────────────────────────
if new_rows:
    new_df = pd.DataFrame(new_rows)
    updated = pd.concat([carn_meta, new_df], ignore_index=True)
    updated.to_csv(CARN_META, index=False)
    print(f'Metadata updated: {len(updated)} total rows → {CARN_META}')

total_carn = len(list(CARN_MIDI.glob('*.mid')))
print(f'Total Carnatic MIDI files now: {total_carn}')

## Cell 6 — Final Summary

In [ ]:
print('=== Indian Classical Transcription Summary ===')
print()

for tradition, midi_dir, meta_path in [
    ('Hindustani', HIND_MIDI, HIND_META),
    ('Carnatic',   CARN_MIDI, CARN_META),
]:
    midi_files = list(midi_dir.glob('*.mid'))
    meta       = pd.read_csv(meta_path) if meta_path.exists() else pd.DataFrame()
    with_midi  = meta['midi_path'].notna().sum() if 'midi_path' in meta.columns else 0
    print(f'{tradition}')
    print(f'  MIDI files in processed dir : {len(midi_files)}')
    print(f'  Rows in metadata CSV        : {len(meta)}')
    print(f'  Rows with midi_path         : {with_midi}')
    print()

print('Next step: re-run the Music Transformer Colab notebook.')
print('The training scripts will automatically pick up all MIDI files in the processed dirs.')